In [ ]:
# %%
import cv2
import numpy as np
import os
import time
import math

In [ ]:
# Methods for default stitching


def find_stitch(prior, current):
    h, w = prior.shape
    prior_mask = (prior == 0).astype(np.uint8)
    current_mask = (current == 0).astype(np.uint8)

    best_score = -1
    best_offset = [22, 55]

    # Loop through all possible offsets
    for y in range(10, 800):
        for x in range(0, 800):
            # Calculate placement range on canvas
            canvas_h = h+100
            canvas_w = w+100

            # Prior will always be placed at bottom-left of canvas
            canvas_prior = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
            canvas_prior[-h:, :w] = prior_mask

            # Calculate position to place the current image
            y_start = canvas_h - h - y #900-800-y(search vertical positions) finds the height of the top left corner of the thing
            x_start = x
            y_end = y_start + h #
            x_end = x_start + w #

            # Skip if current image doesn't fit in canvas
            if y_start < 0 or x_start < 0 or y_end > canvas_h or x_end > canvas_w:
                continue

            canvas_current = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
            canvas_current[y_start:y_end, x_start:x_end] = current_mask

            # Overlap where both have black pixels (1)
            overlap = np.logical_and(canvas_prior, canvas_current)
            score = np.sum(overlap)

            if score > best_score:
                best_score = score
                best_offset = [x, y]

    return best_offset


def brute_force_fit(image_list, index):
    """Optimized stitching function"""
    N = len(image_list)
    offsets = np.zeros((N, 2), dtype=int)
    
    print("Finding optimal stitching offsets...")
    for i in (range(1, N)):
        offsets[i] = find_stitch(image_list[i-1], image_list[i])
    np.savetxt(f"offsets{index}.csv", offsets,delimiter=",",fmt="%d")
    
    # Calculate total canvas size needed
    total_offset = np.cumsum(offsets, axis=0)
    max_offset = np.max(total_offset, axis=0)
    min_offset = np.min(total_offset, axis=0)
    
    canvas_w = image_list[0].shape[1] + (max_offset[0] - min_offset[0]) + 10
    canvas_h = image_list[0].shape[0] + (max_offset[1] - min_offset[1]) + 10
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    
    print("Stitching images...")
    for i in (range(N)):
        img = image_list[i]
        x_offset, y_offset = total_offset[i]
        
        # Calculate positions
        y_pos = canvas_h - img.shape[0] - y_offset
        x_pos = x_offset
        
        # Create mask for dark pixels
        mask = img < 200
        
        # Calculate valid positions
        y_start = max(y_pos, 0)
        y_end = min(y_pos + img.shape[0], canvas_h)
        x_start = max(x_pos, 0)
        x_end = min(x_pos + img.shape[1], canvas_w)
        
        # Only proceed if there's overlap
        if y_start < y_end and x_start < x_end:
            # Calculate corresponding image region
            img_y_start = y_start - y_pos
            img_y_end = img_y_start + (y_end - y_start)
            img_x_start = x_start - x_pos
            img_x_end = img_x_start + (x_end - x_start)
            
            # Apply mask and set pixels
            canvas[y_start:y_end, x_start:x_end] = np.where(
                mask[img_y_start:img_y_end, img_x_start:img_x_end],
                255,
                canvas[y_start:y_end, x_start:x_end]
            )
    
    print("Stitching complete!")
    return canvas


#Functions for different modes

def find_stitch_fig(prior,current):

    h, w = prior.shape
    prior_mask =(prior == 0).astype(np.uint8)
    current_mask=(current == 0).astype(np.uint8)

    
    

    best_score = -1
    best_offset = [22, 55]

    # Loop through all possible offsets
    for y in range(10, 800):
        for x in range(0, 800):
            # Calculate placement range on canvas
            canvas_h = h+100
            canvas_w = w+100

            

            # Calculate position to place the current image
            y_start = 100 - y #900-800-y(search vertical positions) finds the height of the top left corner of the thing
            x_start = x
            y_end = y_start + h #
            #print (float(x)+float(w))
            #if (x+w) >800:
            #    x_end = 800
            #else:
            x_end = x_start + w #

            # Prior will always be placed at bottom-left of canvas
            canvas_prior = np.zeros((canvas_h, canvas_w),dtype=np.uint8)
            
            
            canvas_prior[-h:, :w] = prior_mask

            canvas_prior_exp = (canvas_prior==1)
            prit = np.zeros((900,900),dtype=np.uint8)
            

            # Skip if current image doesn't fit in canvas
            if y_start < 0 or x_start < 0 or y_end > canvas_h or x_end > canvas_w:
                continue

            canvas_current = np.zeros((canvas_h, canvas_w),dtype=np.uint8)
            
            
            canvas_current[y_start:y_end, x_start:x_end] = current_mask
            canvas_current_exp= (canvas_current==1)
            curt = np.zeros((900,900),dtype=np.uint8)
            
            #print(canvas_current)
            # Overlap where both have black pixels (1)
            overlap = np.logical_and(canvas_prior, canvas_current)
            score = np.sum(overlap)

            if score > best_score:
                

                
                over = np.zeros((900,900),dtype=np.uint8)
                for j in range(0,900):
                    for i in range(0,900):
                        
                        if overlap[i,j]:
                            over[i,j] = 0
                        else:
                            over[i,j] = 255

                for j in range(0,900):
                    for i in range(0,900):
                        
                        if canvas_current_exp[i,j]:
                            curt[i,j] = 0
                        else:
                            curt[i,j] = 255

                for j in range(0,900):
                    for i in range(0,900):
                        
                        if canvas_prior_exp[i,j]:
                            prit[i,j] = 0
                        else:
                            prit[i,j] = 255
                pri =prit
                cur=curt


                
                
                best_score = score
                best_offset = [x, y]


    return over,pri,cur


def brute_force_fit_color(image_list, index):
    """Optimized stitching function"""
    N = len(image_list)
    offsets = np.zeros((N, 2), dtype=int)

    filename =f"offsets{index}.csv"
    print("Finding optimal stitching offsets...")
    offsets = np.loadtxt(filename,delimiter=",",dtype=int)
    
    
    # Calculate total canvas size needed
    total_offset = np.cumsum(offsets, axis=0)
    max_offset = np.max(total_offset, axis=0)
    min_offset = np.min(total_offset, axis=0)
    
    canvas_w = image_list[0].shape[1] + (max_offset[0] - min_offset[0]) + 10
    canvas_h = image_list[0].shape[0] + (max_offset[1] - min_offset[1]) + 10
    canvas = np.zeros((canvas_h, canvas_w,3), dtype=np.uint8)
    
    print("Stitching images...")
    for i in (range(N)):
        img = image_list[i]
        x_offset, y_offset = total_offset[i]
        
        # Calculate positions
        y_pos = canvas_h - img.shape[0] - y_offset
        x_pos = x_offset
        
        # Create mask for dark pixels
        
        
        
        mask = img != [255,255,255]
        # Calculate valid positions
        y_start = max(y_pos, 0)
        y_end = min(y_pos + img.shape[0], canvas_h)
        x_start = max(x_pos, 0)
        x_end = min(x_pos + img.shape[1], canvas_w)
        
        # Only proceed if there's overlap
        if y_start < y_end and x_start < x_end:
            # Calculate corresponding image region
            img_y_start = y_start - y_pos
            img_y_end = img_y_start + (y_end - y_start)
            img_x_start = x_start - x_pos
            img_x_end = img_x_start + (x_end - x_start)
            
            for y in range(0,img_y_end-img_y_start):
                for x in range(0,img_x_end-img_x_start):
                    
                    if mask[y,x,0] or mask[y,x,1] or mask[y,x,2]:
                        #canvas[y_start+y, x_start+x] = img[img_y_start+y,img_x_start+x]
                        if canvas[y_start+y, x_start+x,0] == 0 and canvas[y_start+y, x_start+x,1] == 0 and canvas[y_start+y, x_start+x,2] ==0:

                            canvas[y_start+y, x_start+x]=img[img_y_start+y,img_x_start+x]
                        else:
                            canvas[y_start+y, x_start+x]=img[img_y_start+y,img_x_start+x] 
            
    
    print("Stitching complete!")
    return canvas


def brute_force_fit_color_reverse(image_list, index):
    """Optimized stitching function"""
    N = len(image_list)
    offsets = np.zeros((N, 2), dtype=int)

    filename =f"offsets{index}.csv"
    print("Finding optimal stitching offsets...")
    offsets = np.loadtxt(filename,delimiter=",",dtype=int)
    
    
    # Calculate total canvas size needed
    total_offset = np.cumsum(offsets, axis=0)
    max_offset = np.max(total_offset, axis=0)
    min_offset = np.min(total_offset, axis=0)
    
    canvas_w = image_list[0].shape[1] + (max_offset[0] - min_offset[0]) + 10
    canvas_h = image_list[0].shape[0] + (max_offset[1] - min_offset[1]) + 10
    canvas = np.zeros((canvas_h, canvas_w,3), dtype=np.uint8)
    
    print("Stitching images...")
    for i in reversed(range(N)):
        img = image_list[i]
        x_offset, y_offset = total_offset[i]
        
        # Calculate positions
        y_pos = canvas_h - img.shape[0] - y_offset
        x_pos = x_offset
        
        # Create mask for dark pixels
        mask = img != [255,255,255]
        # Calculate valid positions
        y_start = max(y_pos, 0)
        y_end = min(y_pos + img.shape[0], canvas_h)
        x_start = max(x_pos, 0)
        x_end = min(x_pos + img.shape[1], canvas_w)
        
        # Only proceed if there's overlap
        if y_start < y_end and x_start < x_end:
            # Calculate corresponding image region
            img_y_start = y_start - y_pos
            img_y_end = img_y_start + (y_end - y_start)
            img_x_start = x_start - x_pos
            img_x_end = img_x_start + (x_end - x_start)
            
            for y in range(0,img_y_end-img_y_start):
                for x in range(0,img_x_end-img_x_start):
                    
                    if mask[y,x,0] or mask[y,x,1] or mask[y,x,2]:
                        canvas[y_start+y, x_start+x]=img[img_y_start+y,img_x_start+x]
            
    
    print("Stitching complete!")
    return canvas


def brute_force_fit_greyscale(image_list, index):
    """Optimized stitching function"""
    N = len(image_list)
    offsets = np.zeros((N, 2), dtype=int)

    filename =f"offsets{index}.csv"
    print("Finding optimal stitching offsets...")
    offsets = np.loadtxt(filename,delimiter=",",dtype=int)
    
    
    # Calculate total canvas size needed
    total_offset = np.cumsum(offsets, axis=0)
    max_offset = np.max(total_offset, axis=0)
    min_offset = np.min(total_offset, axis=0)
    
    canvas_w = image_list[0].shape[1] + (max_offset[0] - min_offset[0]) + 10
    canvas_h = image_list[0].shape[0] + (max_offset[1] - min_offset[1]) + 10
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    
    print("Stitching images...")
    for i in (range(N)):
        img = image_list[i]
        x_offset, y_offset = total_offset[i]
        
        # Calculate positions
        y_pos = canvas_h - img.shape[0] - y_offset
        x_pos = x_offset
        
        # Create mask for dark pixels
        
        mask = img != 255
        
        # Calculate valid positions
        y_start = max(y_pos, 0)
        y_end = min(y_pos + img.shape[0], canvas_h)
        x_start = max(x_pos, 0)
        x_end = min(x_pos + img.shape[1], canvas_w)
        
        # Only proceed if there's overlap
        if y_start < y_end and x_start < x_end:
            # Calculate corresponding image region
            img_y_start = y_start - y_pos
            img_y_end = img_y_start + (y_end - y_start)
            img_x_start = x_start - x_pos
            img_x_end = img_x_start + (x_end - x_start)
            
            for y in range(0,img_y_end-img_y_start):
                for x in range(0,img_x_end-img_x_start):
                    
                    if mask[y,x]:
                        #canvas[y_start+y, x_start+x] = img[img_y_start+y,img_x_start+x]
                        if canvas[y_start+y, x_start+x] == 0:

                            canvas[y_start+y, x_start+x]=img[img_y_start+y,img_x_start+x]
                        else:
                            canvas[y_start+y, x_start+x]=(img[img_y_start+y,img_x_start+x] + canvas[y_start+y, x_start+x])/2
            
    
    print("Stitching complete!")
    return canvas





Default Individual Stitching

In [ ]:


# %%
# Define your target folder
output_dir = r"Columns" #change directory according to your path
start_time = time.time()
# Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
# Load images efficiently
batchnum=0 #start any batch
for j in range(batchnum,14):
    image_list = []
    
    print(f"Loading images batch number {j}")
    for i in (range(31)):
        index = j * 31 + i                    # Global index (0 to 434)        
        path = f"Pros/pros{index:03d}.png"
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            print(f"Warning: Could not load image {path}")
            continue
        image_list.append(img)
    
    if not image_list:
        print("No images loaded!")
    else:
        # Stitch images
        print(f"stitching {j}")
        stitched_column1 = brute_force_fit(image_list,j)
        filename = f"column{j:03d}.png"     # column000.png ... column013.png
        filepath = os.path.join(output_dir, filename)
        
        # Save result
        cv2.imwrite(filepath, stitched_column1)
        print(f"Saved column{j}.png")

    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    minutes = int(minutes)
    seconds = math.ceil(seconds)

    print(f"Execution time for column{j}.png: {minutes} minutes {seconds} seconds")

For making a figure demonstrating stitching between two images

In [ ]:
path = f"Pros/pros251.png"
prior = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
path = f"Pros/pros252.png"
current = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

over,prior,current=find_stitch_fig(prior,current)

cv2.imwrite("/Figure/Prior/p.png", prior)

cv2.imwrite("/Figure/Current/c.png", current)

cv2.imwrite("/Figure/Overlap/o.png",over)

Making columns by simply overlaying the colored images (required cropped and centered color images)

In [ ]:
# Define your target folder
output_dir = r"ColorCol" #change directory according to your path
start_time = time.time()
# Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
# Load images efficiently
batchnum=0 #start any batch
for j in range(batchnum,14):
    image_list = []
    
    print(f"Loading images batch number {j}")
    for i in (range(31)):
        index = j * 31 + i                    # Global index (0 to 434)        
        path = f"ColorsCrop/cropped{index:03d}.png"
        #remove greyscale for true color
        img = cv2.imread(path)
        
        if img is None:
            print(f"Warning: Could not load image {path}")
            continue
        image_list.append(img)
    
    if not image_list:
        print("No images loaded!")
    else:
        # Stitch images
        print(f"stitching {j}")
        stitched_column1 = brute_force_fit_color(image_list,j)
        filename = f"column{j:03d}.png"     # column000.png ... column013.png
        filepath = os.path.join(output_dir, filename)
        
        # Save result
        cv2.imwrite(filepath, stitched_column1)
        print(f"Saved column{j}.png")

    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    minutes = int(minutes)
    seconds = math.ceil(seconds)

    print(f"Execution time for column{j}.png: {minutes} minutes {seconds} seconds")

Overlaying in reverse order

In [ ]:
# Define your target folder
output_dir = r"ColorCol" #change directory according to your path
start_time = time.time()
# Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
# Load images efficiently
batchnum=0 #start any batch
for j in range(batchnum,14):
    image_list = []
    
    print(f"Loading images batch number {j}")
    for i in (range(31)):
        index = j * 31 + i                    # Global index (0 to 434)        
        path = f"ColorsCrop/cropped{index:03d}.png"
        img = cv2.imread(path)
        
        if img is None:
            print(f"Warning: Could not load image {path}")
            continue
        image_list.append(img)
    
    if not image_list:
        print("No images loaded!")
    else:
        # Stitch images
        print(f"stitching {j}")
        stitched_column1 = brute_force_fit_color_reverse(image_list,j)
        filename = f"column{j:03d}.png"     # column000.png ... column013.png
        filepath = os.path.join(output_dir, filename)
        
        # Save result
        cv2.imwrite(filepath, stitched_column1)
        print(f"Saved column{j}.png")

    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    minutes = int(minutes)
    seconds = math.ceil(seconds)

    print(f"Execution time for column{j}.png: {minutes} minutes {seconds} seconds")

Outputting a greyscale column where the images are averaged over their offset

In [ ]:
# Define your target folder
output_dir = r"ColorCol" #change directory according to your path
start_time = time.time()
# Create the folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
# Load images efficiently
batchnum=0 #start any batch
for j in range(batchnum,14):
    image_list = []
    
    print(f"Loading images batch number {j}")
    for i in (range(31)):
        index = j * 31 + i                    # Global index (0 to 434)        
        path = f"ColorsCrop/cropped{index:03d}.png"
        #remove greyscale for true color
        img = cv2.imread(path,cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            print(f"Warning: Could not load image {path}")
            continue
        image_list.append(img)
    
    if not image_list:
        print("No images loaded!")
    else:
        # Stitch images
        print(f"stitching {j}")
        stitched_column1 = brute_force_fit_greyscale(image_list,j)
        filename = f"column{j:03d}.png"     # column000.png ... column013.png
        filepath = os.path.join(output_dir, filename)
        
        # Save result
        cv2.imwrite(filepath, stitched_column1)
        print(f"Saved column{j}.png")

    end_time = time.time()
    elapsed_time = end_time - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    minutes = int(minutes)
    seconds = math.ceil(seconds)

    print(f"Execution time for column{j}.png: {minutes} minutes {seconds} seconds")